[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zyclasses/AILab/blob/main/Week6_Convolutional_Neural_Networks_starter.ipynb)

# Week 6 Lab: Convolutional Neural Networks

This lab goes with Lecture 10. It is **practice**: nothing to hand in, and it is good preparation for Assignment 4,
which trains CNNs on CIFAR-10. It takes about an hour.

| Exercise | Topic | Time |
|---|---|---|
| 1 | Convolution by hand, then with PyTorch, and the output size | 12 min |
| 2 | Edge filters on a Fashion MNIST image | 8 min |
| 3 | Counting a conv layer's weights, and why stack 3 × 3 filters | 10 min |
| 4 | Max pooling, shifts, and global average pooling | 12 min |
| 5 | A small CNN against an MLP of the same size | 18 min |

**How to use it:** replace every `[YOUR CODE]` with one line of code. Where a step says **predict first**, write your
guess down before you run the cell. Run the cells from top to bottom: later exercises reuse names from earlier ones.

The numbers here are a little different from the slides (other images, filters and layer sizes), so your results
will differ from the lecture's. Small differences from run to run, and from the numbers quoted below, are normal.

**First:** choose *Runtime > Change runtime type > T4 GPU*, then run the setup cell.

## Setup (run this first)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)          # if this says cpu, switch the runtime to a T4 GPU

# Fashion MNIST as images: [N, 1, 28, 28], standardized with the training images' mean and std, kept on the GPU
data = torchvision.datasets.FashionMNIST(root="data", train=True, download=True)
torch.manual_seed(42)
train_part, valid_part = torch.utils.data.random_split(range(60_000), [55_000, 5_000])
images = data.data.float().div(255).unsqueeze(1)        # pixels from 0 to 1, with a channel axis
X_train, y_train = images[train_part.indices], data.targets[train_part.indices]
X_valid, y_valid = images[valid_part.indices], data.targets[valid_part.indices]
mean, std = X_train.mean(), X_train.std()
X_train, X_valid = ((X_train - mean) / std).to(device), ((X_valid - mean) / std).to(device)
y_train, y_valid = y_train.to(device), y_valid.to(device)
xentropy = nn.CrossEntropyLoss()

def batches(X, y, batch_size=128):
    # shuffled mini-batches of tensors that are already on the GPU (a quick DataLoader)
    order = torch.randperm(len(X), device=X.device)
    for i in range(0, len(X), batch_size):
        rows = order[i:i + batch_size]
        yield X[rows], y[rows]

def valid_accuracy(model):
    # accuracy (%) on the 5,000 validation images
    with torch.no_grad():
        return round((model(X_valid).argmax(dim=1) == y_valid).float().mean().item() * 100, 1)

def fit(model, optimizer, n_epochs):
    # Lecture 7's training loop. Returns the validation accuracy after each epoch.
    accuracies = []
    for epoch in range(n_epochs):
        model.train()
        for X_batch, y_batch in batches(X_train, y_train):
            loss = xentropy(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
        model.eval()
        accuracies.append(valid_accuracy(model))
    return accuracies

def n_params(model):
    return sum(p.numel() for p in model.parameters())

print("training images:", tuple(X_train.shape), " validation images:", tuple(X_valid.shape))

---
## Exercise 1: Convolution by hand, then with PyTorch (Lecture 10)

**Step 1.** A 4 × 4 image and a 3 × 3 filter. Slide the filter over every position where it fits. At each one,
multiply the overlapping numbers and add them up.

**Predict first:** what is the output's size? And the top-left output value?

In [ ]:
X = torch.tensor([[1., 2., 0., 1.],
                  [0., 1., 3., 1.],
                  [2., 1., 0., 2.],
                  [1., 0., 1., 3.]])
K = torch.tensor([[1., 0., -1.],
                  [1., 0., -1.],
                  [1., 0., -1.]])

n, k = len(X), len(K)
out = torch.zeros(n - k + 1, n - k + 1)
for i in range(n - k + 1):
    for j in range(n - k + 1):
        [YOUR CODE]  # out[i, j] = ... (the patch at (i, j) times K, added up)
print(out)

by_pytorch = F.conv2d(X[None, None], K[None, None])     # [batch, channels, height, width] for both
print("same as F.conv2d:", torch.equal(out, by_pytorch[0, 0]))

**Step 2.** The output size, for input size $n$, filter size $k$, padding $p$ and stride $s$:
$\left\lfloor \frac{n + 2p - k}{s} \right\rfloor + 1$. Write it as a function, and check it against `nn.Conv2d` on a
28 × 28 image.

**Predict first:** which of the five settings below keeps the size at 28?

**You're done when:** every line prints `True`.

In [ ]:
def out_size(n, k, p, s):
    [YOUR CODE]  # return ... (the formula: // rounds down)

image = torch.rand(1, 1, 28, 28)
for k, p, s in [(3, 0, 1), (3, 1, 1), (5, 2, 1), (5, 2, 2), (7, 0, 3)]:
    real = nn.Conv2d(1, 1, kernel_size=k, padding=p, stride=s)(image).shape[-1]
    print(f"k={k}, p={p}, s={s}: formula {out_size(28, k, p, s)}, nn.Conv2d {real}, {out_size(28, k, p, s) == real}")

---
## Exercise 2: Edge filters on a Fashion MNIST image (Lecture 10)

The filter `K` from Exercise 1 responds where the brightness changes from left to right: a **vertical** edge. Turned on
its side, it finds **horizontal** edges. Let's run both on a pair of trousers.

**Predict first:** which filter responds more to trousers, in total?

**You're done when:** the vertical-edge map lights up along the sides of the legs, and its total is the larger one
(about **480** against **110**).

In [ ]:
k_trousers = (y_valid == 1).nonzero()[0].item()          # the first trousers (class 1) in the validation set
picture = images[valid_part.indices[k_trousers], 0]     # [28, 28], pixels from 0 to 1

vertical = K
[YOUR CODE]  # horizontal = ... (the vertical filter turned on its side)
filters = torch.stack([vertical, horizontal]).unsqueeze(1)      # [2 filters, 1 channel, 3, 3]
[YOUR CODE]  # maps = ... (both filters at once, padding 1, then [0])
print("maps:", tuple(maps.shape))
print(f"total response: vertical {maps[0].abs().sum():.0f}, horizontal {maps[1].abs().sum():.0f}")

fig, axes = plt.subplots(1, 3, figsize=(9, 3))
for ax, im, title in zip(axes, [picture, maps[0].abs(), maps[1].abs()], ["image", "vertical edges", "horizontal edges"]):
    ax.imshow(im, cmap="gray"); ax.set_title(title); ax.axis("off")
plt.show()

*Finished early?* Try a bag (class 8) or a sandal (class 5). Which filter wins now, and why?

---
## Exercise 3: Counting a conv layer's weights (Lecture 10)

**Step 1.** A conv layer has $C_\text{out}$ filters. Each is $k \times k \times C_\text{in}$ weights, plus one bias.
Write the count, and check it against PyTorch's.

**Predict first:** does the count depend on the image size?

In [ ]:
def conv_params(c_in, c_out, k):
    [YOUR CODE]  # return ... (the weights of all the filters, plus one bias each)

for c_in, c_out, k in [(1, 16, 5), (16, 32, 3), (3, 64, 3), (64, 128, 3)]:
    layer = nn.Conv2d(c_in, c_out, kernel_size=k)
    print(f"Conv2d({c_in}, {c_out}, {k}): formula {conv_params(c_in, c_out, k):,}, PyTorch {n_params(layer):,}")

**Step 2.** Two 3 × 3 conv layers in a row see a 5 × 5 patch of their input, just like one 5 × 5 layer. That's why
VGG stacks 3 × 3 filters. Compare their weights, at 64 channels in and out, without biases.

**Predict first:** which uses fewer weights, and about how many times fewer?

**You're done when:** the stack has **73,728** weights against **102,400** (about **1.39** times fewer), and both
see the same **5 × 5** patch.

In [ ]:
one_5x5 = nn.Conv2d(64, 64, kernel_size=5, padding=2, bias=False)
two_3x3 = nn.Sequential(nn.Conv2d(64, 64, kernel_size=3, padding=1, bias=False),
                        nn.Conv2d(64, 64, kernel_size=3, padding=1, bias=False))
[YOUR CODE]  # ratio = ... (how many times more weights the 5 x 5 has)
print(f"one 5 x 5: {n_params(one_5x5):,} weights,  two 3 x 3: {n_params(two_3x3):,} weights,  ratio {ratio:.2f}")

# what each output sees: put a single 1 in the middle of the input, set every weight to 1, and see which outputs light up
dot = torch.zeros(1, 64, 11, 11)
dot[0, :, 5, 5] = 1
with torch.no_grad():
    for name, layer in [("one 5 x 5", one_5x5), ("two 3 x 3", two_3x3)]:
        for p in layer.parameters():
            p.fill_(1.0)
        seen = (layer(dot)[0, 0] != 0).nonzero()
        rows, cols = seen[:, 0], seen[:, 1]
        print(f"{name}: the 1 reaches a {(rows.max() - rows.min() + 1).item()} x {(cols.max() - cols.min() + 1).item()} block of outputs")

---
## Exercise 4: Max pooling, shifts, and global average pooling (Lecture 10)

**Step 1.** Pooling splits each map into 2 × 2 windows (stride 2, the default) and keeps one number per window.

**Predict first:** write down the max-pooled and the average-pooled 2 × 2 results by hand, then check.

In [ ]:
M = torch.tensor([[1., 5., 2., 0.],
                  [3., 2., 8., 4.],
                  [0., 6., 1., 1.],
                  [4., 2., 3., 7.]])
[YOUR CODE]  # max_pool = ... (2 x 2 max pooling)
[YOUR CODE]  # avg_pool = ... (2 x 2 average pooling)
print("max:\n", max_pool(M[None, None])[0, 0])
print("average:\n", avg_pool(M[None, None])[0, 0])

**Step 2.** Shift an image to the right and max-pool it. `torch.roll` moves the pixels and wraps the last columns
around to the left, where Fashion MNIST is blank anyway.

**Predict first:** after a shift of **2** pixels, how is the pooled map related to the pooled map of the original?
After a shift of **1** pixel?

**You're done when:** a 1-pixel shift leaves about **71%** of the pooled values in place, and the last line prints
`True`: after a 2-pixel shift, the pooled map is the old one moved by exactly 1 cell. So max pooling doesn't make the
network ignore shifts. It makes small ones matter less.

In [ ]:
original = X_valid[k_trousers:k_trousers + 1]           # the trousers again, standardized: [1, 1, 28, 28]
pooled = max_pool(original)                             # [1, 1, 14, 14]
for shift in (1, 2):
    [YOUR CODE]  # shifted = ... (move the image right by shift pixels)
    pooled_shifted = max_pool(shifted)
    same_place = (pooled_shifted == pooled).float().mean().item()
    print(f"shift {shift}: {100 * same_place:.0f}% of the pooled values unchanged")
print("2-pixel shift = pooled map moved by 1 cell:",
      torch.equal(max_pool(torch.roll(original, 2, dims=3)), torch.roll(pooled, 1, dims=3)))

**Step 3.** Global average pooling turns each map into one number, whatever its size. It's how the classic CNNs'
big dense layers were replaced.

**You're done when:** the shape is `[8, 64]` and it equals the plain average over the height and the width.

In [ ]:
maps = torch.randn(8, 64, 7, 7)                          # a batch of 8, with 64 maps of 7 x 7
[YOUR CODE]  # gap = ... (global average pooling: output size 1)
pooled = gap(maps).flatten(1)
print(tuple(pooled.shape), torch.allclose(pooled, maps.mean(dim=(2, 3))))

---
## Exercise 5: A small CNN against an MLP of the same size (Lecture 10)

**Step 1.** Complete the CNN: a second conv block (32 filters of 3 × 3, padding 1, ReLU, 2 × 2 max pooling), and the
size of the first dense layer's input.

**Predict first:** the shape after each `MaxPool2d`, and after `Flatten`.

In [ ]:
torch.manual_seed(0)
cnn = nn.Sequential(
    nn.Conv2d(1, 16, kernel_size=5, padding=2), nn.ReLU(), nn.MaxPool2d(2),
    [YOUR CODE]  # the second conv block: Conv2d, ReLU, MaxPool2d
    nn.Flatten(),
    [YOUR CODE]  # nn.Linear(..., 64), nn.ReLU(): what size comes out of Flatten?
    nn.Linear(64, 10),
).to(device)

Z = X_valid[:2]                                          # 2 images
for layer in cnn:
    [YOUR CODE]  # Z = ... (pass Z through this layer)
    print(f"{type(layer).__name__:10s} {tuple(Z.shape)}")
print("parameters:", f"{n_params(cnn):,}")

**Step 2.** An MLP with about the same number of parameters, for a fair comparison. Train both the same way: Adam,
learning rate 0.001, 5 epochs, the same batch order.

**Predict first:** which wins, and by how much?

**You're done when:** the CNN reaches about **91%** and the MLP about **87%**.

In [ ]:
torch.manual_seed(0)
mlp = nn.Sequential(nn.Flatten(),
                    nn.Linear(28 * 28, 128), nn.ReLU(),
                    nn.Linear(128, 64), nn.ReLU(),
                    nn.Linear(64, 10)).to(device)
print(f"CNN parameters: {n_params(cnn):,}   MLP parameters: {n_params(mlp):,}")

for name, model in [("CNN", cnn), ("MLP", mlp)]:
    torch.manual_seed(1)                                 # the same batch order for both
    [YOUR CODE]  # optimizer = ... (Adam, learning rate 0.001)
    print(name, fit(model, optimizer, 5))

**Step 3.** Where are the CNN's parameters? Count the conv layers' and the dense layers' separately.

**You're done when:** the dense layers hold about **95%** of them.

In [ ]:
[YOUR CODE]  # conv_count = ... (the parameters of the Conv2d layers)
dense_count = sum(n_params(m) for m in cnn if isinstance(m, nn.Linear))
print(f"conv layers: {conv_count:,}   dense layers: {dense_count:,} ({100 * dense_count / n_params(cnn):.0f}%)")

*Finished early?* Add `nn.BatchNorm2d(16)` and `nn.BatchNorm2d(32)` after the two conv layers (before the ReLU), and
train again. Or replace `nn.Flatten()` and the first dense layer with `nn.AdaptiveAvgPool2d(1), nn.Flatten(),
nn.Linear(32, 64)`. How many parameters are left, and what happens to the accuracy?

---
## Questions to think about

Nothing to hand in. If you can't answer one in a sentence or two, go back to that part of the slides or this notebook.

1. Why doesn't a conv layer's number of weights depend on the image size, while a dense layer's does?
2. A 3 × 3 conv with padding 1 and stride 2 gets a 28 × 28 map. What size comes out?
3. Max pooling makes small shifts matter less. When does a 1-pixel shift still change the pooled output?
4. The CNN beat an MLP with about the same number of parameters. What does the CNN know about images that the MLP
   doesn't?
5. What was the most confusing part of this week?